## Importing Packages

In [160]:
import pandas as pd
import numpy as np
import polars as pl
import glob

import matplotlib.pyplot as plt
import seaborn as sns

from tvDatafeed import TvDatafeed, Interval
import backtesting

import os

## Creating the Dataset

### Logging into TVFeed

In [161]:
tv = TvDatafeed(username = 'jaganathapandiyan12', password = 'PASS$1234TO5678')

error while signin
you are using nologin method, data you access may be limited


### Downloading the stocks in NIFTY 500 (list downloaded from NSE website on 16th November)

In [162]:
data_nifty500 = pd.read_csv('ind_nifty500list.csv')

symbols_nifty500 = data_nifty500['Symbol'].to_list()
symbols_nifty500 = [s.replace("-", "_") for s in symbols_nifty500]
symbols_nifty500.remove('DUMMYSKFIN')


print(f'Total stocks: {len(symbols_nifty500)}')

Total stocks: 501


In [163]:
failed = []

if not os.path.exists("Data/Processed/Prices_all.parquet"):
    for symbol in symbols_nifty500:
        try:
            df = tv.get_hist(symbol = symbol, exchange = 'NSE', interval = Interval.in_daily, n_bars = 5000)
            df.to_parquet(f"data/raw/prices/{symbol}.parquet")
        
        except Exception as e:
            failed.append(symbol)


    while len(failed) != 0:
        for symbol in failed[:]:
            try:
                df = tv.get_hist(symbol = symbol, exchange = 'NSE', interval = Interval.in_daily, n_bars = 5000)
                df.to_parquet(f"data/raw/prices/{symbol}.parquet")
                failed.remove(symbol)
            
            except:
                pass

    print('\nFailed cases were run repeatedly until success. Data is complete')

else:
    print('Skipping as processed data file already exists')

Skipping as processed data file already exists


### Merging the files into a single file for analysis

In [164]:
RAW_DIR = "Data/Raw/Prices"
OUT_FILE = "Data/Processed/Prices_all.parquet"

def merge_parquet_files():
    if os.path.exists(OUT_FILE):
        print("Skipping to avoid overwrite as merged data file already exists")
        return

    files = glob.glob(os.path.join(RAW_DIR, "*.parquet"))

    if len(files) == 0:
        print("No parquet files found in data/raw/prices/")
        return

    dfs = []

    for f in files:
        symbol = os.path.basename(f).replace(".parquet", "")
        
        df = (pl.read_parquet(f).with_columns([pl.lit(symbol).alias("symbol"), 
                                               pl.col("datetime").cast(pl.Date).alias("date")
                                               ]).drop("datetime"))
        dfs.append(df)

    final_df = pl.concat(dfs, how = "vertical")

    final_df = final_df.sort(["symbol", "date"])

    final_df.write_parquet(OUT_FILE)

    print(f"Master datafile created: {OUT_FILE}")
    print(f"\nRows: {final_df.height}, Columns: {final_df.width}")


if __name__ == "__main__":
    merge_parquet_files()

Skipping to avoid overwrite as merged data file already exists


## Calculating the Required Ratios

In [165]:
df = pl.read_parquet("Data/Processed/Prices_all.parquet")
df = df.sort(["symbol", "date"])

### 1) MACD (12-day EMA - 26-day EMA)

In [167]:
df = df.with_columns([
    pl.col("close").ewm_mean(span = 12).over("symbol").alias("ema_12"),
    pl.col("close").ewm_mean(span = 26).over("symbol").alias("ema_26"),]).with_columns([
    (pl.col("ema_12") - pl.col("ema_26")).alias("macd")])

### 2) 14-day ROC

In [166]:
df = df.with_columns([
    ((pl.col("close") - pl.col("close").shift(14)) 
     / pl.col("close").shift(14)).over("symbol").alias("roc_14")])

### 3) 14-day ADX

In [172]:
df = df.with_columns([
    pl.col("high").shift(1).over("symbol").alias("prev_high"),
    pl.col("low").shift(1).over("symbol").alias("prev_low"),
    pl.col("close").shift(1).over("symbol").alias("prev_close"),])

df = df.with_columns([
    (pl.col("high") - pl.col("low")).alias("hl_range"),
    (pl.col("high") - pl.col("prev_close")).abs().alias("hc_range"),
    (pl.col("low") - pl.col("prev_close")).abs().alias("lc_range"),])

df = df.with_columns([
    pl.max_horizontal([
        pl.col("hl_range"),
        pl.col("hc_range"),
        pl.col("lc_range"),
    ]).alias("tr")])

df = df.with_columns([
    (pl.col("high") - pl.col("prev_high")).alias("up_move"),
    (pl.col("prev_low") - pl.col("low")).alias("down_move"),])

df = df.with_columns([
    pl.when(
        (pl.col("up_move") > pl.col("down_move")) & (pl.col("up_move") > 0)
        ).then(pl.col("up_move")).otherwise(0).alias("plus_dm"),

    pl.when(
        (pl.col("down_move") > pl.col("up_move")) & (pl.col("down_move") > 0)
        ).then(pl.col("down_move")).otherwise(0).alias("minus_dm"),])

df = df.with_columns([
    pl.col("tr").rolling_mean(14).over("symbol").alias("atr_14"),
    pl.col("plus_dm").rolling_mean(14).over("symbol").alias("plus_dm_14"),
    pl.col("minus_dm").rolling_mean(14).over("symbol").alias("minus_dm_14"),])


df = df.with_columns([
    (100 * pl.col("plus_dm_14") / pl.col("atr_14")).alias("plus_di_14"),
    (100 * pl.col("minus_dm_14") / pl.col("atr_14")).alias("minus_di_14"),])


df = df.with_columns([
    (100 * (pl.col("plus_di_14") - pl.col("minus_di_14")).abs()
     / (pl.col("plus_di_14") + pl.col("minus_di_14"))
    ).alias("dx_14")])

df = df.with_columns([
    pl.col("dx_14").rolling_mean(14).over("symbol").alias("adx_14")])

### 4) 5-day VWAP

In [169]:
df = df.with_columns([
    ((pl.col("high") + pl.col("low") + pl.col("close")) / 3).alias("typical_price")])

df = df.with_columns([
    (pl.col("typical_price") * pl.col("volume")).rolling_sum(5).over("symbol").alias("tp_vol_sum_5"),
    pl.col("volume").rolling_sum(5).over("symbol").alias("vol_sum_5"),
                    ]).with_columns([(pl.col("tp_vol_sum_5") / pl.col("vol_sum_5")).alias("vwap_5")])

### 5) 14-day RSI

In [168]:
df = df.with_columns(
    pl.col("close").diff().over("symbol").alias("delta"))

df = df.with_columns([
    pl.col("delta").clip(lower_bound=0).alias("gain"),
    (-pl.col("delta").clip(upper_bound=0)).alias("loss")])

df = df.with_columns([
    pl.col("gain").rolling_mean(14).over("symbol").alias("avg_gain_14"),
    pl.col("loss").rolling_mean(14).over("symbol").alias("avg_loss_14")])

df = df.with_columns((
    100 - 100 / (1 + (pl.col("avg_gain_14") / pl.col("avg_loss_14")))).alias("rsi_14"))

### 6) 20-day Volume

In [171]:
df = df.with_columns(
    pl.col("volume").rolling_mean(20).over("symbol").alias("sma_vol_20"))

### 7) 14-day ATR

In [170]:
df = df.with_columns([
    pl.col("close").shift(1).over("symbol").alias("prev_close")])

df = df.with_columns([
    pl.max_horizontal([pl.col("high") - pl.col("low"),
                       (pl.col("high") - pl.col("prev_close")).abs(),
                       (pl.col("low")  - pl.col("prev_close")).abs()]).alias("true_range")])

df = df.with_columns([
    pl.col("true_range").rolling_mean(14).over("symbol").alias("atr_14")])

### Dropping unnecessary columns and removing NULL rows

In [173]:
df = df.drop(["ema_12", "ema_26", "delta", "gain", "loss", "avg_gain_14", "avg_loss_14", "typical_price", 
              "tp_vol_sum_5", "vol_sum_5", "prev_close", "true_range", "prev_high", "prev_low", "prev_close", 
              "hl_range", "hc_range", "lc_range", "up_move", "down_move", "plus_dm", "minus_dm", "plus_dm_14", 
              "minus_dm_14", "dx_14", "tr", "plus_di_14", "minus_di_14"])

In [ ]:
print(f'Before dropping nulls: {df.shape}')

df = df.drop_nulls()

print(f'After dropping nulls: {df.shape}')

Before dropping nulls: (1732923, 14)
After dropping nulls: (1719921, 14)


In [176]:
df.head()

symbol,open,high,low,close,volume,date,roc_14,macd,rsi_14,vwap_5,atr_14,sma_vol_20,adx_14
str,f64,f64,f64,f64,f64,date,f64,f64,f64,f64,f64,f64,f64
"""360ONE""",328.0125,333.5,328.0,331.8625,165132.0,2019-10-31,0.074119,1.712562,61.791967,326.770211,18.708036,118640.6,24.247002
"""360ONE""",331.25,334.75,330.275,332.7875,96216.0,2019-11-01,0.117299,2.926782,70.099238,328.021379,17.796429,119981.6,25.962721
"""360ONE""",333.0,338.25,315.2125,328.7625,53556.0,2019-11-04,0.137636,3.549972,74.230886,329.395792,18.55,113581.6,26.747462
"""360ONE""",331.75,331.75,318.75,320.725,23288.0,2019-11-05,0.162416,3.411506,79.084861,330.097129,18.211607,112011.0,27.168052
"""360ONE""",333.5,333.5,318.75,322.4625,28080.0,2019-11-06,0.148013,3.39091,78.167344,330.021879,15.672321,111360.6,27.31945


## Setting up the Strategies

### 1) Low Risk-Low Reward Strategy